# Import Libraries

In [1]:
import os
import copy
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [2]:
from tqdm.auto import tqdm
import time

# Device Configuration

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device: {device}")

Using Device: cpu


# Dataset Paths

In [4]:
TRAIN_DIR = Path("../data/raw/Training")
TEST_DIR = Path("../data/raw/Testing")

IMAGE_SIZE = 224
BATCH_SIZE = 32

# Data Transforms

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
from torchvision import datasets
from torch.utils.data import DataLoader

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_dataset.classes

# Model Training

# Load Pretrained EfficientNet-B0

### Load the Model

In [7]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)

print("EfficientNet-B0 loaded successfully.")

EfficientNet-B0 loaded successfully.


In [8]:
#Inspect the Classifier
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [9]:
#Replace the Classifier
num_features = model.classifier[1].in_features

model.classifier[1] = torch.nn.Linear(
    in_features=num_features,
    out_features=4
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)


In [10]:
model = model.to(device)

print(f"Model moved to {device}")

Model moved to cpu


In [11]:
#Verify the Model
print("=" * 50)
print("Model Summary")
print("=" * 50)

print(f"Architecture : EfficientNet-B0")
print(f"Output Classes : {len(class_names)}")
print(f"Device : {device}")

Model Summary
Architecture : EfficientNet-B0
Output Classes : 4
Device : cpu


# Defining Loss Function & Optimizer

In [12]:
#Hyperparameters

LEARNING_RATE = 1e-4
NUM_EPOCHS = 15
WEIGHT_DECAY = 1e-4

print("Learning Rate :", LEARNING_RATE)
print("Epochs        :", NUM_EPOCHS)
print("Weight Decay  :", WEIGHT_DECAY)

Learning Rate : 0.0001
Epochs        : 15
Weight Decay  : 0.0001


In [13]:
#Define Loss Function
criterion = nn.CrossEntropyLoss()

print("Loss Function:", criterion)

Loss Function: CrossEntropyLoss()


In [14]:
#Define Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Optimizer Created Successfully.")

Optimizer Created Successfully.


In [15]:
#Display's Optimizer Details
print("=" * 50)
print("Optimizer Configuration")
print("=" * 50)

print(f"Optimizer      : AdamW")
print(f"Learning Rate  : {LEARNING_RATE}")
print(f"Weight Decay   : {WEIGHT_DECAY}")

Optimizer Configuration
Optimizer      : AdamW
Learning Rate  : 0.0001
Weight Decay   : 0.0001


In [16]:
#Verify Everything
print("=" * 50)
print("Training Configuration")
print("=" * 50)

print(f"Model          : EfficientNet-B0")
print(f"Classes        : {len(class_names)}")
print(f"Loss Function  : {criterion.__class__.__name__}")
print(f"Optimizer      : {optimizer.__class__.__name__}")
print(f"Epochs         : {NUM_EPOCHS}")
print(f"Device         : {device}")

Training Configuration
Model          : EfficientNet-B0
Classes        : 4
Loss Function  : CrossEntropyLoss
Optimizer      : AdamW
Epochs         : 15
Device         : cpu


# Learning Rate Scheduler

In [17]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=2
)

print("Learning Rate Scheduler Created Successfully.")

Learning Rate Scheduler Created Successfully.


In [18]:
print("=" * 50)
print("Scheduler Configuration")
print("=" * 50)

print(f"Scheduler : {scheduler.__class__.__name__}")
print(f"Current LR: {optimizer.param_groups[0]['lr']}")

Scheduler Configuration
Scheduler : ReduceLROnPlateau
Current LR: 0.0001


# Training Function

In [19]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    progress_bar = tqdm(
        dataloader,
        desc="Training",
        leave=False
    )

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        correct_predictions += (predicted == labels).sum().item()

        total_samples += labels.size(0)

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = running_loss / total_samples
    epoch_accuracy = (correct_predictions / total_samples) * 100

    return epoch_loss, epoch_accuracy

# Validation Function

In [20]:
def validate_one_epoch(model, dataloader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    progress_bar = tqdm(
        dataloader,
        desc="Validation",
        leave=False
    )

    with torch.no_grad():

        for images, labels in progress_bar:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            correct_predictions += (predicted == labels).sum().item()

            total_samples += labels.size(0)

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

    epoch_loss = running_loss / total_samples
    epoch_accuracy = (correct_predictions / total_samples) * 100

    return epoch_loss, epoch_accuracy

# Model Training Loop

In [21]:
train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

best_val_accuracy = 0.0

patience = 3
epochs_without_improvement = 0

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = MODEL_DIR / "best_efficientnet_b0.pth"

print("=" * 70)
print("Brain Tumor AI Model Training Started")
print("=" * 70)

overall_start = time.time()

for epoch in range(NUM_EPOCHS):

    epoch_start = time.time()

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_accuracy = validate_one_epoch(
        model,
        test_loader,
        criterion,
        device
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0

        torch.save(model.state_dict(), best_model_path)

        print("Best Model Saved")

    else:

        epochs_without_improvement += 1

    epoch_time = time.time() - epoch_start

    print("-" * 60)
    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_accuracy:.2f}%")
    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Acc  : {val_accuracy:.2f}%")
    print(f"Learning Rate   : {optimizer.param_groups[0]['lr']:.6f}")
    print(f"Epoch Time      : {epoch_time:.2f} sec")
    print("-" * 60)

    if epochs_without_improvement >= patience:

        print("\nEarly Stopping Triggered")
        break

total_time = time.time() - overall_start

print("\n" + "=" * 70)
print("Training Completed Successfully")
print("=" * 70)

print(f"Best Validation Accuracy : {best_val_accuracy:.2f}%")
print(f"Training Time            : {total_time/60:.2f} minutes")
print(f"Best Model Path          : {best_model_path}")

Brain Tumor AI Model Training Started

Epoch [1/15]


Training:   0%|          | 0/175 [00:00<?, ?it/s]

Validation:   0%|          | 0/50 [00:00<?, ?it/s]

Best Model Saved
------------------------------------------------------------
Train Loss      : 0.4119
Train Accuracy  : 87.79%
Validation Loss : 0.2580
Validation Acc  : 92.31%
Learning Rate   : 0.000100
Epoch Time      : 1106.02 sec
------------------------------------------------------------

Epoch [2/15]


Training:   0%|          | 0/175 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/50 [00:00<?, ?it/s]

------------------------------------------------------------
Train Loss      : 0.0267
Train Accuracy  : 99.20%
Validation Loss : 0.2681
Validation Acc  : 95.50%
Learning Rate   : 0.000010
Epoch Time      : 1058.78 sec
------------------------------------------------------------

Epoch [8/15]


Training:   0%|          | 0/175 [00:00<?, ?it/s]

Validation:   0%|          | 0/50 [00:00<?, ?it/s]

------------------------------------------------------------
Train Loss      : 0.0163
Train Accuracy  : 99.57%
Validation Loss : 0.2713
Validation Acc  : 95.56%
Learning Rate   : 0.000010
Epoch Time      : 1422.49 sec
------------------------------------------------------------

Early Stopping Triggered

Training Completed Successfully
Best Validation Accuracy : 95.62%
Training Time            : 150.73 minutes
Best Model Path          : ..\models\best_efficientnet_b0.pth
